--------------------------
#### 오늘의 실습 목표 (타이타닉 버전)
- 결측치/범주형 처리(전처리)
- Pipeline으로 “전처리 + 모델” 한 번에 묶기
- train_test_split(test_size, random_state)가 결과에 미치는 영향 실험
- 모델 파라미터(예: RandomForest의 max_depth)가 과적합에 미치는 영향 실험
- cross_val_score로 성능을 안정적으로 보기
--------------------------

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score

In [ ]:
# 숫자/문자 컬럼 자동 분리
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))  # 결측치: 중앙값
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),  # 결측치: 최빈값
    ("onehot", OneHotEncoder(handle_unknown="ignore"))     # 문자 → 원핫
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

In [ ]:
for test_size in [0.1, 0.2, 0.3, 0.4]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("test_size:", test_size, "acc:", round(acc, 4))

In [ ]:
for rs in [0, 1, 42, 100, 999]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=rs
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print("random_state:", rs, "acc:", round(acc, 4))

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("CV scores:", scores)
print("CV mean:", scores.mean(), "std:", scores.std())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

scores = cross_val_score(rf_clf, X, y, cv=5, scoring="accuracy")
print("RF CV mean:", scores.mean(), "std:", scores.std())

In [ ]:
for depth in [2, 3, 4, 5, 7, 10, None]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=depth,
        random_state=42
    )
    rf_clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", rf)
    ])
    scores = cross_val_score(rf_clf, X, y, cv=5, scoring="accuracy")
    print("max_depth:", depth, "CV mean:", round(scores.mean(), 4))

##### GridSearchCV
- 여러 하이퍼파라미터 조합을 자동으로 전부 실험해서 가장 좋은 조합을 찾아주는 도구


 sklearn 패키지(=라이브러리)
- └── model_selection (모듈)
-              └── GridSearchCV (클래스)
-              └── train_test_split
-              └── KFold
-              └── cross_val_score

In [ ]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf)
])

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5, 7, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

gs = GridSearchCV(rf_clf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X, y)

print("Best score:", gs.best_score_)
print("Best params:", gs.best_params_)